## 03. StateGraph
- StateGraph는 LangGraph의 핵심 구성 요소로, 상태 기반의 워크플로우를 구축하는 데 사용되는 가장 기본적인 그래프 구조
- **그래프는 노드와 엣지로 구성되며, 각 노드는 특정 작업을 수행하고 상태를 업데이트**
- **StateGraph의 주요 특징은 상태를 중심으로 동작**
  - 상태는 `TypedDict`나 `Pydantic BaseModel`을 사용하여 정의되며, 그래프 실행 중 지속적으로 업데이트됨 
  - 노드 간의 전환은 조건부 엣지를 통해 제어될 수 있어, 복잡한 의사 결정 프로세스를 모델링할 수 있음 
  - StateGraph는 재귀적 실행을 지원하여 반복적인 작업을 효과적으로 처리할 수 있음

- **체크포인터를 통해 상태를 저장하고 관리할 수 있어 중단과 재개가 가능한 워크플로우를 구현할 수 가능 $\rightarrow$ StateGraph는 복잡한 AI 시스템, 특히 대화형 에이전트나 다단계 의사 결정 프로세스를 구현하는 데 매우 적합**

<br>

<img src='img/napkin-selection.png' width=400> 

<br>

### 기본 구성 요소

<br>

### State (상태) - 데이터 저장소
- **모든 노드가 공유하는 데이터 저장소**
  - 모든 노드가 같은 State를 보고 수정
  - Python의 `TypedDict`를 사용해서 정의
  - 어떤 종류의 데이터든 저장 가능

<br>



In [ ]:
class SimpleState(TypedDict):
    count: int
    name: str

<br>

#### `MessagesState` : 대화형 앱을 위한 편의 클래스
- LangGraph는 대화형 앱에 자주 사용되는 `MessagesState`를 내장으로 제공

In [1]:
from langgraph.graph import MessagesState

In [2]:
class MyState(MessagesState):
    # messages 필드가 자동 포함 (add_messages 리듀서 적용)
    user_name: str  # 추가 필드도 정의 가능

<br>

#### Node (노드) - 작업하는 함수
- **Node는 실제 작업을 수행하는 Python 함수입니다. 현재 상태를 받아서 새로운 상태 반환**
- **노드의 규칙**:
  - 항상 state를 첫 번째 매개변수
  - 딕셔너리 형태로 새로운 상태를 반환
  - 반환하지 않은 필드는 기존 값을 유지

In [3]:
# 카운터를 1 증가시키는 노드
def add_one(state):
    return {"count": state["count"] + 1}

# 이름을 설정하는 노드  
def set_name(state):
    return {"name": "Alice"}

<br>

#### Edge (엣지) - 연결하는 화살표
- **Edge는 노드들 사이의 연결을 정의**

```python
from langgraph.graph import START, END

# 기본적인 엣지 연결
graph.add_edge("노드1", "노드2")  # 노드1 → 노드2

# 시작과 끝 연결 (START, END 상수 사용)
graph.add_edge(START, "첫번째노드")  # 시작 → 첫번째노드
graph.add_edge("마지막노드", END)   # 마지막노드 → 끝
```

<br>

#### 전체 구조 예시

<img src='img/1-3-1.png' width=150>

In [8]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

- State: 카운터를 저장하는 상자

In [9]:
class CounterState(TypedDict):
    count: int

* Node: 카운터를 증가시키는 함수

In [10]:
def increment(state):
    print(f"현재 카운트: {state['count']}")
    new_count = state["count"] + 1
    print(f"새로운 카운트: {new_count}")
    return {"count": new_count}

- Edge: 노드들을 연결하는 그래프

In [11]:
graph = StateGraph(CounterState)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)

In [13]:
app = graph.compile()
result = app.invoke({"count": 0})
print(f"최종 결과: {result}")

현재 카운트: 0
새로운 카운트: 1
최종 결과: {'count': 1}


<br>

#### 두 개의 노드 연결

- 노드 정의

In [14]:
# 첫 번째 증가 함수
def first_increment(state):
    print("첫 번째 증가")
    return {"count": state["count"] + 1}

# 두 번째 증가 함수  
def second_increment(state):
    print("두 번째 증가")
    return {"count": state["count"] + 10}

- 그래프 구성

In [15]:
graph = StateGraph(CounterState)
graph.add_node("first", first_increment)
graph.add_node("second", second_increment)

- 엣지 구성

In [ ]:
graph.add_edge(START, "first")
graph.add_edge("first", "second") 
graph.add_edge("second", END)

In [17]:
app = graph.compile()
result = app.invoke({"count": 0})
print(f"최종 결과: {result}")

첫 번째 증가
두 번째 증가
최종 결과: {'count': 11}


<br>

### 상태 (State)
- **LangGraph에서 상태는 그래프의 전체 데이터 흐름을 관리하는 핵심 요소**
- 주로 TypedDict나 Pydantic BaseModel을 사용하여 정의
- **그래프의 모든 노드와 엣지에 대한 입력 스키마 역할**

<br>

#### 리듀서와 `add_messages`
- **상태 필드에 리듀서(Reducer)를 지정하면 노드가 반환한 값을 기존 상태에 어떻게 적용할지 정의 가능**
  -  대화형 앱에서는 `add_messages` 리듀서가 표준 패턴

<br>

```python
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # 메시지 자동 누적
    user_name: str                            # 일반 필드 (덮어쓰기)
```

<br>

- **`add_messages`는 새 메시지를 기존 리스트에 추가하고, 같은 ID의 메시지는 업데이트하는 리듀서**

In [18]:
from langgraph.graph import MessagesState

class MyState(MessagesState):
    # messages 필드 + add_messages 리듀서가 자동 포함
    user_name: str

<br>

### 상태 관리
- LangGraph에서 상태 관리는 AI 시스템의 핵심
- 상태(State)는 시스템이 처리하는 모든 정보를 담는 그릇이며, 노드 간에 데이터를 전달하고 축적하는 역할

<br>

#### 상태의 본질
- LangGraph는 Google의 Pregel 시스템에서 영감을 받은 메시지 전달(Message Passing) 아키텍처를 사용

1. 노드는 독립적인 처리 단위: 각 노드는 독립적으로 실행되며, 서로 직접 통신하지 않음
2. 상태를 통한 간접 통신: 노드들은 공유 상태를 통해 데이터를 주고받음
3. Super-step 실행 모델: 병렬 실행 가능한 노드들은 같은 super-step에서 동시에 실행

<br>

#### 상태의 역할
- 간단한 상태 예시
  
```python
class ChatState(TypedDict):
    user_message: str      # 노드가 처리할 현재 메시지. 각 노드 실행 시 덮어쓰기됩니다
    chat_history: list     # 대화의 연속성을 위해 누적되는 리스트
    user_context: dict     # 사용자별 설정, 선호도 등을 저장하는 유연한 구조
    system_status: str     # 워크플로우의 현재 상태를 추적 (예: "processing", "waiting", "completed")

```

<br>

### 상태 관리의 중요성
- 복잡한 대화 흐름에서 맥락을 일관되게 유지하여 사용자 경험을 향상
- **일관성(Consistency) : 대화나 작업의 전체 흐름에서 정보가 손실되지 않고 유지되는 것을 의미**
  - 예) 사용자가 "내 이름은 홍길동입니다"라고 한 후 "내 이름이 뭐라고 했죠?"라고 물으면 시스템이 이를 기억

<br>


####  일관성 유지
- 대화 맥락 유지 예시
  - `def conversation_node(state: ChatState):` : 첫 번째 인자로 현재 상태를 받으며, 선택적으로 두 번째 인자로 config를 받을 수 있음
  - `state["chat_history"][-3:]` : 너무 많은 컨텍스트는 LLM의 성능을 저하시키고 비용을 증가시키며, 3개는 일반적으로 충분한 컨텍스트를 제공하는 균형점

```python
def conversation_node(state: ChatState):
    # 이전 대화 내용을 참조하여 일관된 응답 생성
    context = "\n".join(state["chat_history"][-3:])  # 최근 3개 대화
    response = generate_contextual_response(state["user_message"], context)
    return {"ai_response": response}
```

<br>

#### 맥락 이해 향상
- 축적된 정보를 바탕으로 더 정확하고 맥락에 맞는 응답을 생성
- 맥락 이해 : 단순한 키워드 매칭을 넘어서는 것
  - 예) "그것을 예약해주세요"라는 요청이 있을 때, "그것"이 이전 대화에서 언급된 "오후 3시 회의실"을 의미한다는 것을 이해하는 능력


<br>

#### 복잡한 작업 처리
- 복잡한 작업은 여러 단계로 구성:
  - 예) "항공권 예약" 작업:
    - 1.날짜 확인 $\rightarrow$ 2. 목적지 선택 $\rightarrow$ 3. 좌석 선택 $\rightarrow$ 4. 결제 처리
    - 각 단계의 결과가 다음 단계의 입력이 되며, 상태 관리가 이를 가능



<br>

#### 오류 복구 및 중단점 관리
- 체크포인팅(Checkpointing)은 프로그램의 현재 상태를 저장하는 기술
- 문제가 발생했을 때 마지막 체크포인트부터 다시 시작


<br>

#### 성능 최적화
- 상태에 계산 결과를 캐싱하면 동일한 계산을 반복하지 않아도 됨
  - 예) API 호출 결과를 상태에 저장하면 같은 요청에 대해 재호출할 필요가 없음


<br>



### 기본 상태 구조

In [19]:
from typing_extensions import TypedDict
from typing import Annotated, List
from operator import add

In [20]:
class BasicState(TypedDict):
    # 단순 값들 - 덮어쓰기 방식
    current_step: str
    user_id: str

    # 누적되는 값들 - 추가 방식
    messages: Annotated[List[str], add]
    processing_log: Annotated[List[str], add]

<br>

#### 리듀서(Reducer)
- **리듀서는 함수형 프로그래밍 개념으로, "기존 값과 새 값을 어떻게 합칠 것인가"를 정의하는 함수**
  - JavaScript의 Redux나 React의 useReducer와 유사한 개념

<br>

1. **기본 리듀서 (덮어쓰기)**
- 새 값이 들어오면 기존 값은 완전히 사라짐
- 가장 단순하고 직관적인 방식

```python
current_step: str
user_id: str
```

<br>

```python
# 초기 상태
state = {"current_step": "시작", "user_id": "user123"}

# 노드 1 실행
update1 = {"current_step": "처리중"}
# 결과: {"current_step": "처리중", "user_id": "user123"}

# 노드 2 실행  
update2 = {"current_step": "완료"}
# 결과: {"current_step": "완료", "user_id": "user123"}
```

<br>

2. **커스텀 리듀서 (`add`를 사용한 누적)**

```python
messages: Annotated[List[str], add]
processing_log: Annotated[List[str], add]
```

- **`Annotated` 타입** : Python 3.9에서 도입된 타입 힌팅 기능
  - 첫 번째 인자: 실제 타입 (List[str])
  - 두 번째 인자: 메타데이터 (리듀서 함수 등)

In [22]:
from operator import add

In [ ]:
# 리스트의 경우
result = add([1, 2], [3, 4])  # [1, 2, 3, 4]

# 문자열의 경우
result = add("Hello", " World")  # "Hello World"

# 숫자의 경우
result = add(5, 3)  # 8

<br>

#### 적용 예시
- 초기 상태

In [ ]:
state = {
    "messages": ["안녕하세요"],
    "processing_log": ["시작"]
}

- 노드 1 실행
  - 업데이트 후 상태
    - `messages`: ["안녕하세요", "어떻게 도와드릴까요?"]
    - `processing_log`: ["시작", "사용자 인사 감지"]

In [24]:
def node1(state):
    return {
        "messages": ["어떻게 도와드릴까요?"],
        "processing_log": ["사용자 인사 감지"]
    }

- 노드 2 실행
  - 최종 상태
    - `messages`: ["안녕하세요", "어떻게 도와드릴까요?", "무엇을 도와드릴까요?"]
    - `processing_log`: ["시작", "사용자 인사 감지", "응답 생성 완료"]

In [26]:
def node2(state):
    return {
        "messages": ["무엇을 도와드릴까요?"],
        "processing_log": ["응답 생성 완료"]
    }

<br>

### 상태 스키마 설계
- 상태 스키마는 LangGraph에서 데이터 흐름의 청사진
- 절한 스키마 설계는 시스템의 안정성과 유지보수성을 크게 향상

<br>

#### `TypedDict` (권장)
- 가장 일반적이고 권장되는 방법
- **장점**:
  - 가볍고 빠른 성능
  - Python 표준 라이브러리만 사용
  - LangGraph의 기본 스키마 방식
  - 런타임 오버헤드 없음
- **단점**:
  - 런타임 유효성 검사 없음
  - 기본값 설정이 불편함
  - **`NotRequired`로 선택적 필드 정의**

In [27]:
from typing_extensions import TypedDict, NotRequired
from typing import Annotated
from operator import add

In [28]:
class MyState(TypedDict):
    query: str                          # 필수 필드
    results: Annotated[list[str], add]  # 필수 (리듀서 적용)
    count: NotRequired[int]             # 선택적 필드 (노드에서 생략 가능)


<br>

#### `dataclass`
- **장점**:
  - 기본값 설정이 쉬움
  - 자동 `init`, `repr` 생성
  - TypedDict와 유사한 성능

- **단점**:
  - 런타임 유효성 검사 없음

In [29]:
from dataclasses import dataclass, field
from typing import Annotated
from operator import add

In [ ]:
@dataclass
class MyState:
    query: str = ""
    results: Annotated[list[str], add] = field(default_factory=list)
    count: int = 0

<br>

#### `Pydantic BaseModel`
- 엄격한 데이터 검증이 필요할 때 사용
- **장점**
  - 런타임 데이터 검증
  - 풍부한 유효성 검사 옵션
  - 자동 타입 변환
- **단점**
  - `TypedDict`보다 느림 (검증 오버헤드)
  - 추가 의존성 필요


In [31]:
from pydantic import BaseModel, Field
from typing import Annotated
from operator import add

In [32]:
class MyState(BaseModel):
    query: str = ""
    results: Annotated[list[str], add] = Field(default_factory=list)
    count: int = Field(default=0, ge=0)  # 0 이상만 허용

<br>

#### 상태 스키마 설계 선택 가이드

<table>
<thead>
<tr>
<th>상황</th>
<th>추천 방법</th>
</tr>
</thead>
<tbody>
<tr>
<td>일반적인 경우</td>
<td>TypedDict</td>
</tr>
<tr>
<td>기본값 필요</td>
<td>dataclass</td>
</tr>
<tr>
<td>엄격한 검증 필요</td>
<td>Pydantic</td>
</tr>
<tr>
<td>고성능 필요</td>
<td>TypedDict</td>
</tr>
<tr>
<td>필드별 선택 여부 구분</td>
<td>TypedDict + NotRequired</td>
</tr>
</tbody>
</table>

<br>

### 스키마 설계 원칙

<br>

#### 명확성 원칙
- **각 필드의 목적과 사용법이 명확하게**

  - 의미 있는 필드명 사용: `data` -> `user_profile`
  - 타입 힌팅 활용: `messages: list -> messages: List[str]`
  - 문서화 추가: `docstring`이나 주석으로 각 필드 설명

<br>

#### 캡슐화 원칙
- **내부 처리용 데이터와 외부 인터페이스를 구분**
- LangGraph에서는 입력/출력 스키마를 분리하여 내부 구현을 숨기고 명확한 인터페이스를 제공

<br>

#### 단일 책임 원칙
- **각 상태 스키마는 하나의 명확한 목적을 가져야 함. 하나의 스키마가 너무 많은 책임을 가지면 복잡도가 증가하고 변경이 어려움**

<br>


### 상태 스키마의 유형

<br>

#### 기본 스키마 (Single Schema)
- **입력과 출력이 동일한 단일 스키마를 사용하는 가장 기본적인 형태**
  - 간단한 챗봇이나 Q&A 시스템
  - 상태가 복잡하지 않은 선형적 워크플로우
  - 프로토타이핑이나 학습 목적

In [33]:
from typing_extensions import TypedDict
from typing import Annotated
from operator import add

In [34]:
class BasicState(TypedDict):
    user_input: str
    ai_response: str
    conversation_history: Annotated[list[str], add]

In [35]:
def chatbot_node(state: BasicState) -> dict:
    response = f"'{state['user_input']}'에 대한 응답입니다."
    return {
        "ai_response": response,
        "conversation_history": [f"User: {state['user_input']}", f"AI: {response}"]
    }

<br>

#### 명시적 입출력 스키마 (Explicit Input/Output Schema)
- **입력과 출력을 별도로 정의하여 인터페이스를 명확하게 제어**
  - API 같은 명확한 인터페이스가 필요한 경우
  - 외부 시스템과 연동할 때
  - 내부 처리 과정을 숨기고 싶을 때

<br>


In [39]:
class InputState(TypedDict):
    question: str
    
class OutputState(TypedDict):
    answer: str

class OverallState(InputState, OutputState):
    intermediate_data: str
    search_results: list[str]

In [ ]:
def search_node(state: InputState) -> dict:
    return {
        "search_results": ["결과1", "결과2"],
        "intermediate_data": f"'{state['question']}' 검색 완료"
    }

def answer_node(state: OverallState) -> OutputState:
    return {"answer": f"검색 결과: {state['search_results'][0]}"}


- StateGraph 설정

In [41]:
builder = StateGraph(
   OverallState,
   input_schema=InputState,
   output_schema=OutputState
)

<br>

#### 다중 스키마 + PrivateState
- **내부 노드 간 통신을 위한 private 스키마를 포함하는 복잡한 구조**
  - RAG 파이프라인
  - 다단계 처리가 필요한 복잡한 시스템
  - 각 단계별로 다른 데이터 구조가 필요한 경우
- **노드는 그래프 초기화 시 정의되지 않은 새로운 상태 채널(PrivateState의 필드)을 추가할 수 있음 $\rightarrow$ 이를 통해 내부 처리 데이터를 외부에 노출하지 않으면서 노드 간에 전달할 수 있음**

In [42]:
class OverallState(TypedDict):
    question: str
    answer: str

class PrivateState(TypedDict):
    internal_score: float
    debug_info: str

In [43]:
def internal_node(state: OverallState) -> PrivateState:
    # 노드는 OverallState에 정의되지 않은 필드도 반환 가능
    return {
        "internal_score": 0.95,
        "debug_info": "내부 처리 완료"
    }


<br>

<hr>